# KNN Classification for Student Performance Prediction

This notebook uses K-Nearest Neighbors (KNN) to predict student pass/fail status, including feature scaling and optimal K selection.

## Import Libraries

Import necessary libraries for data processing, visualization, and machine learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Enable inline plotting
%matplotlib inline

## Load and Explore Dataset

Load the dataset and examine the first few rows.

In [ ]:
df = pd.read_csv("StudentPerformance.csv")

print("First 5 rows:")
print(df.head())

## Data Preprocessing

Convert categorical data and create binary target variable.

In [ ]:
# Convert categorical to numerical
df['Extracurricular Activities'] = df['Extracurricular Activities'].map({'Yes': 1, 'No': 0})

# Create binary target: Pass if Performance Index >= 60
df['Pass'] = (df['Performance Index'] >= 60).astype(int)

# Drop original performance index
df.drop('Performance Index', axis=1, inplace=True)

print("Preprocessing completed.")

## Exploratory Data Analysis

Visualize relationships between features and the target variable.

In [ ]:
# Pairplot with hue for Pass/Fail
sns.pairplot(df, hue='Pass')
plt.title('Pairplot of Features Colored by Pass Status')
plt.show()

## Feature Scaling

Standardize features for KNN algorithm.

In [ ]:
# Scale features
scaler = StandardScaler()
scaler.fit(df.drop('Pass', axis=1))
scaled_features = scaler.transform(df.drop('Pass', axis=1))

# Create scaled dataframe
df_feat = pd.DataFrame(scaled_features, columns=df.columns[:-1])
print("Scaled features:")
print(df_feat.head())

## Model Training and Initial Evaluation

Split data and train KNN with K=1, then evaluate.

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(scaled_features, df['Pass'], test_size=0.30, random_state=101)

# Train KNN with K=1
knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(X_train, y_train)
pred = knn.predict(X_test)

print("Initial KNN (K=1) Results:")
print("Confusion Matrix:")
print(confusion_matrix(y_test, pred))
print("\nClassification Report:")
print(classification_report(y_test, pred))

## Finding Optimal K Value

Test different K values and plot error rates to find the best K.

In [ ]:
# Calculate error rates for K=1 to 40
error_rate = []
for i in range(1, 40):
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train, y_train)
    pred_i = knn.predict(X_test)
    error_rate.append(np.mean(pred_i != y_test))

# Plot error rate vs K
plt.figure(figsize=(10, 6))
plt.plot(range(1, 40), error_rate, color='blue', linestyle='dashed', marker='o',
         markerfacecolor='red', markersize=10)
plt.title('Error Rate vs. K Value')
plt.xlabel('K')
plt.ylabel('Error Rate')
plt.show()

# Find best K
best_k = error_rate.index(min(error_rate)) + 1
print(f'Best K value: {best_k}')

## Final Model with Optimal K

Train and evaluate KNN using the optimal K value.

In [ ]:
# Train with best K
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train, y_train)
pred = knn.predict(X_test)

print(f'KNN Results with Optimal K={best_k}:')
print("Confusion Matrix:")
print(confusion_matrix(y_test, pred))
print("\nClassification Report:")
print(classification_report(y_test, pred))